# Clase 25: Detección de Placas + OCR

**Diplomado en Data Science Aplicada con Python** · Arca Continental Ecuador x UDLA

---

**Objetivos de hoy:**
1. Recordar transfer learning de la clase 24 (Coca vs Pepsi).
2. Conocer las **5 tareas** de visión por computadora.
3. Aprender qué es **detección** y cómo funciona YOLO.
4. Probar **YOLO26 pretrained** en imágenes propias.
5. Hacer **fine-tuning** sobre datos de placas etiquetados.
6. Construir el pipeline completo: detección → OCR → texto.
7. Conectar todo a una app Gradio.

## 0. Imports y entorno

> **Importante:** este notebook está pensado para **Colab con GPU**.
> Activar GPU: *Runtime → Change runtime type → T4 GPU* antes de ejecutar.

In [ ]:
!pip install -q ultralytics easyocr cvat-sdk gradio

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import urllib.request, io, os, zipfile, time

import torch
from ultralytics import YOLO
import cv2
from PIL import Image

print(f"PyTorch: {torch.__version__}")
print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

---
## 1. Recap clase 24: transfer learning + Coca vs Pepsi

Antes de meternos en detección, recordemos qué es transfer learning. **La idea es exactamente la misma** que vamos a usar en YOLO, solo cambia la arquitectura.

### La idea visual

> Una CNN preentrenada (ej. VGG16, entrenada en ImageNet con 1.4M de imágenes) tiene filtros que ya saben detectar bordes, texturas, formas. **Reusamos esos filtros congelados** y solo entrenamos un cabezal Dense con nuestras pocas fotos.

### Por qué funciona

- **Capas tempranas**: bordes, manchas, gradientes simples (universales).
- **Capas medias**: texturas, esquinas, patrones repetitivos.
- **Capas tardías**: partes (orejas, ruedas, ojos) y luego objetos.
- Solo las últimas son específicas de la tarea → reusamos las primeras y reentrenamos las últimas.

### Código que ya conocen (de clase 24)

In [ ]:
# Si quieres correr esto, asegurate de tener X_train, y_train del proyecto Coca vs Pepsi
# Esta celda es solo de referencia, no hace falta ejecutarla aquí.

# from tensorflow.keras.applications import VGG16
# from tensorflow.keras import Sequential, layers
#
# base = VGG16(input_shape=(96,96,3), include_top=False, weights="imagenet")
# base.trainable = False
#
# modelo = Sequential([
#     base,
#     layers.Flatten(),
#     layers.Dense(64, activation="relu"),
#     layers.Dense(2, activation="softmax"),
# ])
# modelo.compile("adam", "sparse_categorical_crossentropy", ["accuracy"])
# modelo.fit(X_train, y_train, epochs=5, validation_split=0.1)

print("Recap clase 24 → ahora extendemos la idea a detección.")

### Limitación de esa aproximación

El modelo de Coca vs Pepsi es de **clasificación**: dice "esta foto es Coca" o "es Pepsi". **Una respuesta para toda la imagen.**

- ¿Y si tengo varias latas en una foto? **No las puede contar.**
- ¿Y si quiero saber dónde está cada una? **No las puede ubicar.**
- ¿Y si necesito leer texto en la lata? **Tampoco.**

Para todo eso hace falta otra tarea: **detección de objetos**. Vamos.

---
## 2. Las 5 tareas de visión por computadora

| Tarea | Salida | Pregunta que responde |
|-------|--------|-----------------------|
| **Clasificación** | 1 etiqueta por imagen | ¿Qué hay en la imagen? |
| **Detección** | N bounding boxes + clase | ¿Qué hay y *dónde*? |
| **Segmentación** | Máscara por pixel | ¿Qué *pixeles* son cada cosa? |
| **Pose / keypoints** | Puntos clave conectados | ¿Dónde están las articulaciones? |
| **OCR** | Texto extraído | ¿Qué *dice* la imagen? |

Hoy nos enfocamos en **detección + OCR** (cascada): caso más común en industria (placas, fechas, lotes, QR, códigos de barras).

---
## 3. Bounding boxes y IoU

Una bounding box son **5 números**: `(x, y, w, h, clase)`. La salida de un detector es una *lista variable* de bboxes — una por objeto detectado.

### IoU (Intersection over Union)

Métrica para comparar caja predicha vs caja real:

$$\text{IoU} = \frac{\text{área de intersección}}{\text{área de unión}}$$

- IoU = 0 → cajas separadas
- IoU = 1 → cajas idénticas
- IoU ≥ 0.5 → convención "detección correcta" (mAP@0.5)

In [ ]:
# Calcular IoU manualmente entre dos cajas
def iou(box_a, box_b):
    """
    box_a, box_b: (x1, y1, x2, y2) — corners
    """
    xa = max(box_a[0], box_b[0])
    ya = max(box_a[1], box_b[1])
    xb = min(box_a[2], box_b[2])
    yb = min(box_a[3], box_b[3])
    inter = max(0, xb - xa) * max(0, yb - ya)
    area_a = (box_a[2]-box_a[0]) * (box_a[3]-box_a[1])
    area_b = (box_b[2]-box_b[0]) * (box_b[3]-box_b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0

# Test
gt = (100, 100, 200, 200)            # caja real
pred1 = (110, 110, 210, 210)          # casi exacta
pred2 = (150, 150, 250, 250)          # solapada
pred3 = (300, 300, 400, 400)          # separada

for name, p in [("casi exacta", pred1), ("solapada", pred2), ("separada", pred3)]:
    print(f"  {name:15s} IoU = {iou(gt, p):.3f}")

---
## 4. Probar YOLO26 pretrained

YOLO26 (Ultralytics, enero 2026) es la última versión estable. Mismo API que YOLOv8/v11: `from ultralytics import YOLO`.

### Cargar el modelo nano (~3M parámetros, corre en CPU)

In [ ]:
# Cargar pretrained en COCO (80 clases genéricas)
model = YOLO("yolo26n.pt")

print(f"Modelo cargado: {model.model_name}")
print(f"Clases ({len(model.names)}): {list(model.names.values())[:15]}...")

# OJO: 'license_plate' NO está en COCO. Por eso necesitaremos fine-tune.

In [ ]:
# Probar con una imagen de calle (cualquier URL pública con autos)
URL = "https://raw.githubusercontent.com/cmosquerat/arca-diplomado/main/clase-25/fig_hero_plate.png"

# Ejecutar inferencia
results = model(URL, save=False, verbose=False)
result = results[0]

# Visualizar
img_with_boxes = result.plot()         # array BGR con cajas dibujadas
img_with_boxes_rgb = cv2.cvtColor(img_with_boxes, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 6))
plt.imshow(img_with_boxes_rgb)
plt.axis("off")
plt.title(f"YOLO26 pretrained — {len(result.boxes)} detecciones")
plt.tight_layout(); plt.show()

In [ ]:
# Inspeccionar las detecciones
boxes = result.boxes
print(f"Detecciones: {len(boxes)}")
for i, box in enumerate(boxes):
    cls_id = int(box.cls)
    cls_name = result.names[cls_id]
    conf = float(box.conf)
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    print(f"  {i}: {cls_name:15s} conf={conf:.2f}  bbox=({x1:.0f},{y1:.0f})-({x2:.0f},{y2:.0f})")

# Notar: el modelo detectó 'car' pero NO 'license_plate' (no está en COCO).
# Por eso necesitamos fine-tune.

---
## 5. Descargar el dataset sin etiquetar

Para que cada estudiante etiquete su lote, hosteamos 100 imágenes sin labels en el repo. Las descargamos:

In [ ]:
# Descargar el ZIP de imágenes sin etiquetar
URL_ZIP = "https://raw.githubusercontent.com/cmosquerat/arca-diplomado/main/clase-25/plates_unlabeled.zip"
LOCAL_ZIP = "plates_unlabeled.zip"

if not os.path.exists(LOCAL_ZIP):
    print("Descargando...")
    urllib.request.urlretrieve(URL_ZIP, LOCAL_ZIP)
print(f"Tamaño: {os.path.getsize(LOCAL_ZIP)/1024/1024:.1f} MB")

# Extraer
with zipfile.ZipFile(LOCAL_ZIP) as z:
    z.extractall(".")

img_dir = Path("plates_unlabeled")
imgs = sorted(img_dir.glob("*.jpg"))
print(f"Imágenes descargadas: {len(imgs)}")
print(f"Ejemplo: {imgs[0]}")

In [ ]:
# Ver algunas muestras del dataset
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, fp in zip(axes.flat, imgs[:8]):
    img = Image.open(fp)
    ax.imshow(img); ax.set_title(fp.name, fontsize=9); ax.axis("off")
plt.suptitle("Muestras del dataset sin etiquetar (descargar y etiquetar en CVAT)",
             fontweight="bold")
plt.tight_layout(); plt.show()

---
## 6. Etiquetar en CVAT.ai (proceso fuera del notebook)

**Pasos manuales en CVAT.ai:**

1. **Crear cuenta** en `app.cvat.ai` (gratis).
2. **Crear nueva *task***: "placas-[tu_apellido]".
3. **Definir 1 label**: `placa` (sin atributos, color rojo).
4. **Subir el ZIP** completo del dataset (CVAT lo descomprime).
5. Abrir la tarea y empezar a dibujar:
   - Atajo `N` para nueva caja
   - Click + arrastrar sobre la placa
   - `Ctrl+S` guarda
   - Flecha derecha avanza imagen
6. **Exportar**: menú *Project → Export task dataset → YOLO 1.1 → Download*

### Conexión CVAT → Python (CVAT-SDK)

Una vez etiquetadas las imágenes en CVAT, podemos descargar el dataset directo desde Python (sin descargar manualmente):

In [ ]:
# Demo de CVAT-SDK (REQUIERE credenciales propias en CVAT.ai)
# Comentado por defecto, descomentar y poner tu user/password

# from cvat_sdk import Client
#
# with Client("https://app.cvat.ai") as client:
#     client.login(("tu_usuario", "tu_password"))
#     task = client.tasks.retrieve(TASK_ID)             # ID de tu task
#     task.export_dataset(
#         format_name="YOLO 1.1",
#         filename="my_labeled_dataset.zip",
#         include_images=True,
#     )
# print("Dataset descargado de CVAT.ai")
print("CVAT-SDK demo (necesita credenciales reales para correr)")

**Alternativa más simple**: el botón "Export" de la UI de CVAT te baja un ZIP. Súbelo al notebook directamente con `files.upload()` de Colab.

### Estructura esperada del ZIP exportado por CVAT (formato YOLO 1.1)

```
labels/
├── img_001.txt
├── img_002.txt
├── ...
obj.names           # nombres de las clases
obj.data            # config
train.txt           # lista de imágenes
```

Cada `.txt` tiene una línea por bbox: `<clase> <cx> <cy> <w> <h>` (normalizados 0-1).

---
## 7. Preparar dataset para entrenar (formato YOLO)

> **Para esta demo en clase**, simulamos que ya tenemos un dataset etiquetado descargando uno público pre-armado. En la próxima clase usaremos el dataset etiquetado por todos.

In [ ]:
# DEMO: descargar un dataset público de placas ya etiquetado en formato YOLO
# Usamos un dataset pequeño de Roboflow Universe (License Plates), pre-formateado.

# Para esta demo, asumimos un dataset YOLO ya estructurado en una URL.
# Si vas a correr esto en Colab, considera el dataset oficial de Ultralytics:
import urllib.request

DEMO_URL = "https://github.com/ultralytics/yolov5/releases/download/v1.0/license-plate-dataset-yolo.zip"

# Si la URL no está disponible o falla, podemos generar un mini-dataset sintético
# para la demo. Documentamos ambos caminos:
print("Para fine-tune real, usar:")
print("  1. Dataset etiquetado por los estudiantes en CVAT (próxima clase)")
print("  2. Demo: subset de License Plate Dataset (Roboflow) o similar")
print("  3. Estructura YOLO esperada:")
print('''
dataset/
├── images/
│   ├── train/img_001.jpg, img_002.jpg, ...
│   └── val/img_101.jpg, img_102.jpg, ...
├── labels/
│   ├── train/img_001.txt, img_002.txt, ...
│   └── val/img_101.txt, img_102.txt, ...
└── data.yaml
''')

In [ ]:
# El archivo data.yaml describe el dataset a Ultralytics:
data_yaml_content = '''
path: /content/dataset                # ruta absoluta
train: images/train                   # imágenes de train
val:   images/val                     # imágenes de val

names:
  0: placa                            # solo 1 clase
'''
print(data_yaml_content)
print("Guardar este contenido en `dataset/data.yaml` antes de entrenar.")

---
## 8. Fine-tune YOLO26 con el dataset etiquetado

Idea: misma del transfer learning de clase 24, pero aplicado a YOLO. El modelo `yolo26n.pt` viene con pesos preentrenados en COCO; los reusamos como punto de partida y entrenamos *unas pocas* epochs sobre nuestras placas.

In [ ]:
# Pseudo-código del fine-tune (descomentar cuando tengas dataset etiquetado)

# model = YOLO("yolo26n.pt")           # cargar pretrained
#
# # Fine-tune
# results = model.train(
#     data="dataset/data.yaml",
#     epochs=30,
#     imgsz=640,
#     batch=16,
#     name="placas_run1",               # nombre para guardar
# )
#
# # Resultados
# print(f"mAP@0.5      = {results.box.map50:.3f}")
# print(f"mAP@0.5:0.95 = {results.box.map:.3f}")
# print(f"Precision    = {results.box.mp:.3f}")
# print(f"Recall       = {results.box.mr:.3f}")
print("Pseudo-código del fine-tune. Lo correremos la próxima clase con el dataset consolidado.")

### ¿Qué espero ver?

| Métrica | Pretrained (sin fine-tune) | Después de fine-tune |
|---------|----------------------------|------------------------|
| mAP@0.5 sobre placas | **0.0** (no detecta) | **>0.85** |
| Tiempo de entrenamiento | 0 | ~10-15 min en GPU T4 |
| Tamaño del modelo | 6 MB | 6 MB (mismo) |

**Insight clave**: el fine-tune no cambia la arquitectura ni el tamaño. Solo ajusta los pesos para que detecte la nueva clase.

---
## 9. OCR: leer el texto de la placa detectada

Una vez que YOLO detectó la placa, **recortamos** la región y la pasamos por un OCR (`EasyOCR` o `Tesseract`) para leer el texto.

> Este es el patrón "**cascade pipeline**": dos modelos especializados encadenados, cada uno hace una cosa muy bien.

In [ ]:
import easyocr

# Inicializar EasyOCR (descarga modelos la primera vez)
reader = easyocr.Reader(['en', 'es'], gpu=torch.cuda.is_available())
print("EasyOCR listo")

In [ ]:
# Pipeline cascada completo
def detect_and_read(image_path, yolo_model, ocr_reader):
    """
    Pipeline: detectar placa con YOLO -> recortar -> leer texto con OCR.

    Returns:
        list of dicts: [{'bbox': (x1,y1,x2,y2), 'conf': 0.95, 'text': 'PCJ-3421'}]
    """
    img = cv2.imread(image_path) if isinstance(image_path, str) else image_path
    if isinstance(img, str):
        # URL fallback
        import urllib
        resp = urllib.request.urlopen(image_path)
        arr = np.asarray(bytearray(resp.read()), dtype=np.uint8)
        img = cv2.imdecode(arr, cv2.IMREAD_COLOR)

    # 1. Detectar
    results = yolo_model(img, verbose=False)
    boxes = results[0].boxes

    detections = []
    for box in boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        conf = float(box.conf)

        # 2. Recortar
        crop = img[y1:y2, x1:x2]
        if crop.size == 0: continue

        # 3. Leer texto
        text_results = ocr_reader.readtext(crop, detail=0, paragraph=False)
        text = " ".join(text_results) if text_results else "(no leído)"

        detections.append({
            "bbox": (x1, y1, x2, y2),
            "conf": conf,
            "text": text,
        })
    return detections

# Por ahora, con yolo26n.pt pretrained, no detectará placas (no está en COCO).
# Pero podemos probar con la clase 'car' como demo:
print("Pipeline definido. Lo corremos completo cuando tengamos el modelo fine-tuneado.")

---
## 10. App Gradio: poner todo junto

Pipeline completo en una interfaz web.

In [ ]:
import gradio as gr

def predict_full_pipeline(image):
    """Recibe imagen (numpy array RGB), devuelve imagen anotada + tabla de placas leídas."""
    if image is None:
        return None, None
    # Convertir RGB -> BGR para OpenCV
    img_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    # Pipeline (con el modelo fine-tuneado en producción)
    detections = detect_and_read(img_bgr, model, reader)

    # Dibujar resultados sobre la imagen
    img_out = img_bgr.copy()
    rows = []
    for i, d in enumerate(detections):
        x1, y1, x2, y2 = d["bbox"]
        cv2.rectangle(img_out, (x1, y1), (x2, y2), (0, 255, 0), 3)
        label = f"{d['text']} ({d['conf']:.2f})"
        cv2.putText(img_out, label, (x1, y1 - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        rows.append([i+1, d['text'], f"{d['conf']:.2f}"])

    img_rgb = cv2.cvtColor(img_out, cv2.COLOR_BGR2RGB)
    df = pd.DataFrame(rows, columns=["#", "Texto leído", "Confianza"])
    return img_rgb, df

with gr.Blocks(title="LPR — Detección y lectura de placas") as demo:
    gr.Markdown("# Detección y Lectura de Placas Vehiculares")
    gr.Markdown("Sube una foto de un vehículo. El modelo detecta la placa y lee el texto.")
    with gr.Row():
        inp = gr.Image(sources=["upload", "webcam"], type="numpy", label="Imagen")
        with gr.Column():
            out_img = gr.Image(label="Resultado")
            out_table = gr.Dataframe(headers=["#", "Texto", "Confianza"], label="Placas detectadas")
    inp.change(predict_full_pipeline, inputs=inp, outputs=[out_img, out_table])

# Para lanzar (descomentar en Colab):
# demo.launch(share=True)
print("App Gradio definida. Llamar a demo.launch(share=True) para iniciarla.")

---
## 11. Tu tarea para la próxima clase

1. **Descargar** `plates_unlabeled.zip` (ya lo hicimos arriba — guarda tu copia local).
2. **Crear cuenta** en `app.cvat.ai`.
3. **Crear task** "placas-[tu_apellido]" con el ZIP, label única `placa`.
4. **Etiquetar** todas las imágenes de tu lote (asignación al final de la clase).
5. **Exportar** formato YOLO 1.1.
6. **Subir** el ZIP exportado al Drive compartido como `[apellido]_etiquetas.zip`.
7. **Deadline**: 24h antes de la próxima clase.

### Tips finales para etiquetar bien

- Caja **ajustada** al rectángulo de la placa (sin margen).
- Etiquetar **todas las placas legibles** en cada imagen.
- **Saltar** imágenes con placas ilegibles / muy lejanas.
- **Consistencia** entre lotes — etiquetas malas estropean el modelo.

### ¿Qué haremos la próxima clase?

1. Consolidar los lotes de todos
2. Fine-tune YOLO26n con el dataset combinado
3. Evaluar mAP, precision, recall
4. Integrar EasyOCR para leer el texto
5. Desplegar la app Gradio con link público
6. Cada quien presenta su modelo en 2-3 minutos

---

## Resumen

| Concepto | Detalle |
|----------|---------|
| Tareas de visión | Clasificación, detección, segmentación, pose, OCR |
| Bounding box | (x, y, w, h, clase) — 5 números |
| IoU | Métrica de calidad de bbox; ≥0.5 es estándar |
| YOLO26 | Última versión (Ultralytics, ene 2026); API idéntica a v8 |
| Familia n→x | Trade-off velocidad vs precisión |
| Fine-tuning | Misma idea de clase 24, ahora con YOLO |
| CVAT.ai | Estándar industrial para etiquetar; SDK Python disponible |
| OCR cascade | Detector → recorte → reconocedor de texto |
| Gradio | App web rápida para demo |